In [5]:
import pandas as pd
import os

file_MYRIAD = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/myriad-hes.csv'
output_dir = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/monthly_chunks'
os.makedirs(output_dir, exist_ok=True)

CHUNK_SIZE = 500_000  # rows per read batch

# Track open file handles so we can append across chunks
file_handles = {}

try:
    for chunk in pd.read_csv(file_MYRIAD, chunksize=CHUNK_SIZE, parse_dates=['starttime']):
        # Keep only 2023-2024 rows
        mask = (chunk['starttime'].dt.year >= 2016) & (chunk['starttime'].dt.year <= 2017)
        subset = chunk.loc[mask].copy()
        if subset.empty:
            continue

        # Group by year-month and append to the corresponding output file
        subset['_ym'] = subset['starttime'].dt.to_period('M')
        for period, group in subset.groupby('_ym'):
            out_path = os.path.join(output_dir, f"myriad_{period}.csv")
            write_header = not os.path.exists(out_path)
            group.drop(columns='_ym').to_csv(out_path, mode='a', header=write_header, index=False)

    print("Done. Monthly files written to:", output_dir)
    print("Files created:", sorted(os.listdir(output_dir)))

finally:
    for fh in file_handles.values():
        fh.close()


Done. Monthly files written to: /Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/monthly_chunks
Files created: ['myriad_2016-01.csv', 'myriad_2016-02.csv', 'myriad_2016-03.csv', 'myriad_2016-04.csv', 'myriad_2016-05.csv', 'myriad_2016-06.csv', 'myriad_2016-07.csv', 'myriad_2016-08.csv', 'myriad_2016-09.csv', 'myriad_2016-10.csv', 'myriad_2016-11.csv', 'myriad_2016-12.csv', 'myriad_2017-01.csv', 'myriad_2017-02.csv', 'myriad_2017-03.csv', 'myriad_2017-04.csv', 'myriad_2017-05.csv', 'myriad_2017-06.csv', 'myriad_2017-07.csv', 'myriad_2017-08.csv', 'myriad_2017-09.csv', 'myriad_2017-10.csv', 'myriad_2017-11.csv', 'myriad_2017-12.csv']


In [1]:
file_monthly_check = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/monthly_chunks/myriad_2016-09.csv'

In [2]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import plotly.express as px
from shapely import wkt
from shapely.geometry import shape

# Resolve the monthly file path robustly (supports relative file_monthly_check)
candidate_paths = []
if 'file_monthly_check' in globals():
    candidate_paths.append(Path(file_monthly_check))
if 'output_dir' in globals() and 'file_monthly_check' in globals():
    candidate_paths.append(Path(output_dir) / Path(file_monthly_check).name)

existing = [p for p in candidate_paths if p.exists()]
if not existing:
    raise FileNotFoundError(f"Could not find file_monthly_check in: {[str(p) for p in candidate_paths]}")

monthly_path = existing[0]
print(f"Using monthly file: {monthly_path}")

df = pd.read_csv(monthly_path)
print("Columns:", df.columns.tolist())


def geom_to_lon_lat(value):
    if pd.isna(value):
        return np.nan, np.nan

    # Already numeric tuple/list style strings: "lon,lat"
    s = str(value).strip()
    m = re.search(r"^\s*([-+]?\d*\.?\d+)\s*,\s*([-+]?\d*\.?\d+)\s*$", s)
    if m:
        return float(m.group(1)), float(m.group(2))

    try:
        # WKT path (POINT/POLYGON/MULTIPOLYGON/etc.)
        geom = wkt.loads(s)
    except Exception:
        geom = None

    if geom is None:
        try:
            # GeoJSON path ({"type":..., "coordinates":...})
            obj = json.loads(s)
            geom = shape(obj)
        except Exception:
            return np.nan, np.nan

    if geom.is_empty:
        return np.nan, np.nan

    # Use representative point so polygons/multipolygons map to an interior location.
    p = geom.representative_point() if geom.geom_type in {"Polygon", "MultiPolygon"} else geom.centroid
    return float(p.x), float(p.y)


# Find likely geometry column names dynamically.
geometry_candidates = [
    c
    for c in df.columns
    if any(k in c.lower() for k in ["geom", "geometry", "wkt", "polygon", "multipolygon"])
]

if {'longitude', 'latitude'}.issubset(df.columns):
    df['lon'] = pd.to_numeric(df['longitude'], errors='coerce')
    df['lat'] = pd.to_numeric(df['latitude'], errors='coerce')
elif {'lon', 'lat'}.issubset(df.columns):
    df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
elif geometry_candidates:
    geom_col = geometry_candidates[0]
    print(f"Using geometry column: {geom_col}")
    lon_lat = df[geom_col].apply(geom_to_lon_lat)
    df['lon'] = lon_lat.map(lambda x: x[0])
    df['lat'] = lon_lat.map(lambda x: x[1])
else:
    raise ValueError(
        "No coordinate columns found. Expected longitude/latitude, lon/lat, or a geometry-like column containing WKT/GeoJSON."
    )

plot_df = df.dropna(subset=['lat', 'lon']).copy()
plot_df = plot_df[(plot_df['lat'].between(-90, 90)) & (plot_df['lon'].between(-180, 180))]

if plot_df.empty:
    raise ValueError("No valid coordinates extracted from geometries in this file.")

print(f"Rows in file: {len(df):,}")
print(f"Rows with valid coordinates: {len(plot_df):,}")

fig = px.density_mapbox(
    plot_df,
    lat='lat',
    lon='lon',
    radius=18,
    zoom=4,
    center=dict(lat=float(plot_df['lat'].mean()), lon=float(plot_df['lon'].mean())),
    mapbox_style='carto-positron',
    title=f"Location density heatmap (polygon-aware): {monthly_path.name}",
)
fig.update_layout(height=700, margin=dict(l=10, r=10, t=55, b=10))
fig.show()

plot_df['lat_bin'] = np.floor(plot_df['lat']).astype(int)
plot_df['lon_bin'] = np.floor(plot_df['lon']).astype(int)
region_counts = (
    plot_df.groupby(['lat_bin', 'lon_bin'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
print('\nTop 10 one-degree bins by record count:')
print(region_counts.head(10).to_string(index=False))

Using monthly file: /Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/monthly_chunks/myriad_2016-09.csv
Columns: ['Event', 'Hazard', 'code', 'starttime', 'endtime', 'Intensity', 'Unit', 'Geometry']
Using geometry column: Geometry
Rows in file: 1,333
Rows with valid coordinates: 1,333


/var/folders/g_/2j33jfhn72j448cr82hzgr0c0002d4/T/ipykernel_61818/950960024.py:93: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(



Top 10 one-degree bins by record count:
 lat_bin  lon_bin  count
     -22      -58    253
      30       80     50
     -15      -48     45
      -7      -45     34
     -22      -61     29
      54      102     24
      17      -91     23
     -27       26     20
     -18       21     20
      29       76     14


In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from shapely import wkt
from shapely.geometry import shape

# Use the monthly file already chosen above.
if 'file_monthly_check' not in globals():
    raise ValueError("file_monthly_check is not defined. Run Cell 2 first.")

candidate_paths = [Path(file_monthly_check)]
if 'output_dir' in globals():
    candidate_paths.append(Path(output_dir) / Path(file_monthly_check).name)

existing = [p for p in candidate_paths if p.exists()]
if not existing:
    raise FileNotFoundError(f"Could not find file_monthly_check in: {[str(p) for p in candidate_paths]}")

monthly_path = existing[0]
poly_df = pd.read_csv(monthly_path)

# Identify a geometry-like column.
geometry_candidates = [
    c for c in poly_df.columns
    if any(k in c.lower() for k in ["geom", "geometry", "wkt", "polygon", "multipolygon"])
]
if not geometry_candidates:
    raise ValueError("No geometry-like column found for polygon plotting.")
geom_col = geometry_candidates[0]
print(f"Using geometry column: {geom_col}")


def parse_geometry(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    try:
        return wkt.loads(s)
    except Exception:
        try:
            return shape(json.loads(s))
        except Exception:
            return None

poly_df['geom_obj'] = poly_df[geom_col].apply(parse_geometry)
poly_df = poly_df[poly_df['geom_obj'].notna()].copy()
poly_df = poly_df[poly_df['geom_obj'].apply(lambda g: g.geom_type in ['Polygon', 'MultiPolygon'])].copy()

if poly_df.empty:
    raise ValueError("No Polygon/MultiPolygon geometries were parsed from this file.")

# Keep each row as a separate feature (no dissolve/merge).
poly_df = poly_df.reset_index(drop=True)
poly_df['feature_id'] = poly_df.index.astype(str)
poly_df['draw_value'] = 1

geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": row['feature_id'],
            "properties": {"feature_id": row['feature_id']},
            "geometry": row['geom_obj'].__geo_interface__,
        }
        for _, row in poly_df.iterrows()
    ],
}

fig_polys = px.choropleth_mapbox(
    poly_df,
    geojson=geojson,
    locations='feature_id',
    featureidkey='properties.feature_id',
    color='draw_value',
    color_continuous_scale=['#2E8B57', '#2E8B57'],
    opacity=0.35,
    mapbox_style='carto-positron',
    zoom=3.25,
    center={'lat': 39.5, 'lon': -98.35},
    hover_name='feature_id',
    title=f"All polygons (unmerged), CONUS view: {monthly_path.name}",
)
fig_polys.update_traces(marker_line_width=0.4, marker_line_color='black')
fig_polys.update_layout(height=760, margin=dict(l=8, r=8, t=55, b=8), coloraxis_showscale=False)
fig_polys.show()

print(f"Polygon features drawn (unmerged): {len(poly_df):,}")

### Understanding the multi-hazard events in North America in 2024

In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from shapely import wkt
from shapely.geometry import shape
from shapely.ops import unary_union

# Build event table for the year available in Myriad monthly chunks.
TARGET_YEAR = 2017

MONTHLY_DIR = Path("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/candidate_multihazards_data/monthly_chunks")
files_for_year = sorted(MONTHLY_DIR.glob(f"myriad_{TARGET_YEAR}-*.csv"))

if not files_for_year:
    available_years = sorted({p.name.split("_")[1].split("-")[0] for p in MONTHLY_DIR.glob("myriad_*.csv")})
    raise FileNotFoundError(
        f"No monthly files found for {TARGET_YEAR} in {MONTHLY_DIR}. Available years: {available_years}"
    )

required_cols = ["Event", "Hazard", "code", "starttime", "endtime", "Geometry"]


def parse_geometry(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if not s:
        return None
    try:
        return wkt.loads(s)
    except Exception:
        try:
            return shape(json.loads(s))
        except Exception:
            return None


def hazard_bucket(hazard_name):
    h = str(hazard_name).strip().lower()

    # Exclude tropical cyclone-focused hazards as requested.
    if any(k in h for k in ["hurricane", "tropical cyclone", "tropical storm", "typhoon", "cyclone"]):
        return None

    if any(k in h for k in ["wildfire", "forest fire", "brush fire", "bushfire", "fire weather"]):
        return "wildfire"

    if any(k in h for k in ["drought", "drough", "flood", "flash flood", "rain", "precip", "pluvial"]):
        return "precipitation"

    if any(k in h for k in ["tornado", "gust", "extreme wind", "straight-line wind", "derecho", "wind"]):
        return "wind"

    if any(k in h for k in ["heat", "cold", "freeze", "frost", "temperature", "heatwave", "cold wave"]):
        return "temperature"

    return None


frames = []
for fp in files_for_year:
    m = pd.read_csv(fp, usecols=required_cols, low_memory=False)
    m["start_dt"] = pd.to_datetime(m["starttime"], errors="coerce", utc=True)
    m = m[m["start_dt"].dt.year == TARGET_YEAR].copy()
    if m.empty:
        continue

    m["hazard_category"] = m["Hazard"].map(hazard_bucket)
    m = m[m["hazard_category"].notna()].copy()
    if m.empty:
        continue

    m["geom_obj"] = m["Geometry"].map(parse_geometry)
    m["source_file"] = fp.name
    frames.append(m)

if not frames:
    raise ValueError(
        f"No {TARGET_YEAR} records matched requested hazard families (wildfire, precipitation, wind excluding tropical cyclones, temperature)."
    )

df_year_filtered = pd.concat(frames, ignore_index=True)

event_rows = []
for event_name, g in df_year_filtered.groupby("Event", dropna=False):
    cats = sorted(set(g["hazard_category"].astype(str)))
    if len(cats) < 2:
        continue

    raw_hazards = sorted(set(g["Hazard"].astype(str)))
    codes = sorted(set(g["code"].dropna().astype(str)))

    geoms = [geom for geom in g["geom_obj"] if geom is not None and not geom.is_empty]
    if geoms:
        try:
            merged = unary_union(geoms)
            pt = merged.representative_point()
            centroid_lat = float(pt.y)
            centroid_lon = float(pt.x)
        except Exception:
            centroid_lat = np.nan
            centroid_lon = np.nan
    else:
        centroid_lat = np.nan
        centroid_lon = np.nan

    event_rows.append({
        "start_date": g["start_dt"].min().date().isoformat(),
        "end_date": g["start_dt"].max().date().isoformat(),
        "event": str(event_name),
        "hazards_included": ", ".join(cats),
        "raw_hazards": " | ".join(raw_hazards),
        "location_code": " | ".join(codes),
        "centroid_lat": centroid_lat,
        "centroid_lon": centroid_lon,
        "n_records": int(len(g)),
        "source_files": " | ".join(sorted(set(g["source_file"])))
    })

events_multihazard = pd.DataFrame(event_rows).sort_values(["start_date", "event"]).reset_index(drop=True)

out_path = Path(f"/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/data/myriad_{TARGET_YEAR}_multihazard_event_list.csv")
events_multihazard.to_csv(out_path, index=False)

print(f"Monthly files scanned for {TARGET_YEAR}: {len(files_for_year)}")
print(f"Filtered records in requested hazard families: {len(df_year_filtered):,}")
print(f"Multi-hazard events found (>=2 requested hazard categories): {len(events_multihazard):,}")
print(f"Saved event list: {out_path}")

events_multihazard.head(100)

Monthly files scanned for 2017: 12
Filtered records in requested hazard families: 20,865
Multi-hazard events found (>=2 requested hazard categories): 9,923
Saved event list: /Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/data/myriad_2017_multihazard_event_list.csv


,start_date,end_date,event,hazards_included,raw_hazards,location_code,centroid_lat,centroid_lon,n_records,source_files
0,2017-01-01,2017-01-28,event121168,"precipitation, wildfire",drought | wildfire,dr6975 | wf1080772,13.9220,21.4405,2,myriad_2017-01.csv
1,2017-01-01,2017-01-30,event121169,"precipitation, wildfire",drought | wildfire,dr6975 | wf1080769,13.9220,21.4405,2,myriad_2017-01.csv
2,2017-01-01,2017-02-28,event121171,"precipitation, temperature",drought | heatwave,dr6984 | hw148606,-31.8220,-65.8165,2,myriad_2017-01.csv | myriad_2017-02.csv
3,2017-01-01,2017-01-23,event121172,"precipitation, wildfire",drought | wildfire,dr6983 | wf1079654,-21.8780,-55.8445,2,myriad_2017-01.csv
4,2017-01-01,2017-01-03,event121173,"temperature, wildfire",heatwave | wildfire,hw147030 | wf1321634,20.3625,-101.5540,2,myriad_2017-01.csv
...,...,...,...,...,...,...,...,...,...,...
95,2017-01-03,2017-01-06,event121264,"temperature, wildfire",heatwave | wildfire,hw147735 | wf1083143,8.1240,26.7315,2,myriad_2017-01.csv
96,2017-01-03,2017-01-06,event121265,"temperature, wildfire",heatwave | wildfire,hw147735 | wf1082969,8.1240,26.7315,2,myriad_2017-01.csv
97,2017-01-03,2017-01-06,event121266,"temperature, wildfire",heatwave | wildfire,hw147735 | wf1083092,8.1240,26.7315,2,myriad_2017-01.csv
98,2017-01-03,2017-01-06,event121267,"temperature, wildfire",heatwave | wildfire,hw147735 | wf1083099,8.1240,26.7315,2,myriad_2017-01.csv


In [5]:
import numpy as np
import pandas as pd
import plotly.express as px

if "events_multihazard" not in globals() or events_multihazard is None or len(events_multihazard) == 0:
    raise ValueError("events_multihazard is not available. Run the 2017 event-table cell first.")

ev = events_multihazard.copy()
ev["start_dt"] = pd.to_datetime(ev["start_date"], errors="coerce")
ev["end_dt"] = pd.to_datetime(ev["end_date"], errors="coerce")
ev["centroid_lat"] = pd.to_numeric(ev["centroid_lat"], errors="coerce")
ev["centroid_lon"] = pd.to_numeric(ev["centroid_lon"], errors="coerce")
ev["n_records"] = pd.to_numeric(ev["n_records"], errors="coerce").fillna(1)

# Approximate CONUS bounds
ev = ev[
    ev["centroid_lat"].between(24.0, 50.0)
    & ev["centroid_lon"].between(-125.0, -66.0)
    & ev["start_dt"].notna()
    & ev["end_dt"].notna()
].copy()

if ev.empty:
    raise ValueError("No events remain after CONUS filter. Check centroid parsing or bounds.")

ev["start_week"] = ev["start_dt"] - pd.to_timedelta(ev["start_dt"].dt.weekday, unit="D")
min_week = ev["start_week"].min()
max_week = ev["end_dt"].max() - pd.to_timedelta(ev["end_dt"].max().weekday(), unit="D")
week_starts = pd.date_range(min_week, max_week, freq="7D")

rows = []
for _, r in ev.iterrows():
    w0 = r["start_dt"] - pd.to_timedelta(r["start_dt"].weekday(), unit="D")
    w1 = r["end_dt"] - pd.to_timedelta(r["end_dt"].weekday(), unit="D")
    for wk in week_starts[(week_starts >= w0) & (week_starts <= w1)]:
        rows.append({
            "week_start": wk,
            "week_label": wk.strftime("%Y-%m-%d"),
            "event": r["event"],
            "hazards_included": r["hazards_included"],
            "start_date": r["start_date"],
            "end_date": r["end_date"],
            "location_code": r.get("location_code", ""),
            "lat": r["centroid_lat"],
            "lon": r["centroid_lon"],
            "n_records": r["n_records"],
        })

weekly_events = pd.DataFrame(rows).sort_values(["week_start", "event"]).reset_index(drop=True)
weekly_events["marker_size"] = np.clip(np.sqrt(weekly_events["n_records"]).astype(float), 4, 20)

fig_weekly = px.scatter_geo(
    weekly_events,
    lat="lat",
    lon="lon",
    scope="usa",
    projection="albers usa",
    animation_frame="week_label",
    color="hazards_included",
    size="marker_size",
    size_max=20,
    opacity=0.75,
    hover_name="event",
    hover_data={
        "start_date": True,
        "end_date": True,
        "hazards_included": True,
        "location_code": True,
        "n_records": True,
        "lat": ":.3f",
        "lon": ":.3f",
        "marker_size": False,
        "week_label": False,
    },
    title="CONUS Multi-Hazard Events (2017) - Weekly Slider"
 )

fig_weekly.update_layout(
    legend_title_text="Hazards included",
    height=760,
    margin=dict(l=20, r=20, t=60, b=20),
    geo=dict(bgcolor="rgba(0,0,0,0)"),
 )

fig_weekly.show()

print(f"Weekly frames: {weekly_events['week_label'].nunique()}")
print(f"Event-week points plotted: {len(weekly_events):,}")

Weekly frames: 48
Event-week points plotted: 496
